# 5.2 - Adult Counterfactual Sampling using DiCE (Modular API)

This notebook tests the DiCE-style method implemented in the modular `counterfactuals` framework.

Goals:
- load the trained Adult classifier through the common model adapter
- downsample training data with k-medoids for faster iteration
- fit and run `DiceMethod`
- evaluate with shared modular metrics

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
from sklearn_extra.cluster import KMedoids

from training.lit_classifier import LitClassifier
from training.datamodules.adult import (
    CARDINALITIES,
    INPUT_TYPES,
)
from models.classifiers import TabularClassifier

from counterfactuals.core.base_classes import CounterfactualExample
from counterfactuals.datasets.loaders import AdultDataset
from counterfactuals.methods.dice import DiceMethod
from counterfactuals.metrics.plausibility import KNNPlausibility
from counterfactuals.metrics.proximity import L1Proximity, L2Proximity
from counterfactuals.metrics.sparsity import SparsityMetric
from counterfactuals.metrics.validity import ValidityMetric
from counterfactuals.models.torch_model import TorchModelWrapper

DEVICE = 'cpu'
CKPT = '../checkpoints/adult_classifier/last-v3.ckpt'

In [2]:
# Load classifier and wrap it with the common model interface
backbone = TabularClassifier(
    input_types=INPUT_TYPES,
    cardinalities=CARDINALITIES,
    embedding_dim=1,
    hidden_dims=[64, 32],
    num_classes=2,
)

lit = LitClassifier.load_from_checkpoint(CKPT, model=backbone, map_location=DEVICE)
model = lit.model.eval().to(DEVICE)
model_api = TorchModelWrapper(model=model, device=DEVICE)

print(f'embed_dim : {model.embed_dim}')
print(f'net       : {model.net}')

embed_dim : 14
net       : Sequential(
  (0): Linear(in_features=14, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=2, bias=True)
)


In [3]:
# Load Adult data
adult = AdultDataset(data_dir='../data/', seed=42)
adult.load()

x_train_np, y_train_np = adult.get_train()
x_test_np, y_test_np = adult.get_test()

print(f'Train: {x_train_np.shape}, Test: {x_test_np.shape}')

Train: (39074, 14), Test: (4884, 14)


In [4]:
# Downsample training data with k-medoids for faster implementation testing
x_train_full_np = x_train_np
y_train_full_np = y_train_np

MEDOID_TARGET_SIZE = 3000
KMEDOIDS_FIT_SAMPLE_SIZE = 8000
rng = np.random.default_rng(42)

n_train = x_train_full_np.shape[0]
sample_size = min(KMEDOIDS_FIT_SAMPLE_SIZE, n_train)
n_medoids = min(MEDOID_TARGET_SIZE, sample_size)

if n_medoids < n_train:
    sampled_indices_parts = []
    for cls in np.unique(y_train_full_np):
        cls_idx = np.where(y_train_full_np == cls)[0]
        cls_take = max(1, int(round(sample_size * (len(cls_idx) / n_train))))
        cls_take = min(cls_take, len(cls_idx))
        sampled_indices_parts.append(rng.choice(cls_idx, size=cls_take, replace=False))

    sampled_indices = np.unique(np.concatenate(sampled_indices_parts))
    x_sample = x_train_full_np[sampled_indices]
    y_sample = y_train_full_np[sampled_indices]

    kmedoids = KMedoids(
        n_clusters=n_medoids,
        metric='euclidean',
        method='alternate',
        init='k-medoids++',
        random_state=42,
    )
    kmedoids.fit(x_sample)

    medoid_local_idx = np.sort(kmedoids.medoid_indices_)
    x_train_np = x_sample[medoid_local_idx]
    y_train_np = y_sample[medoid_local_idx]

print(f'Full train: {x_train_full_np.shape} -> Downsampled train: {x_train_np.shape}')
print(f'Test remains unchanged: {x_test_np.shape}')

Full train: (39074, 14) -> Downsampled train: (3000, 14)
Test remains unchanged: (4884, 14)


## Fit DiCE Method

In [5]:
dice = DiceMethod(
    total_cfs=4,
    proximity_weight=0.5,
    diversity_weight=1.0,
    yloss_type='hinge_loss',
    learning_rate=0.05,
    min_iter=100,
    max_iter=300,
    stopping_threshold=0.5,
    random_seed=42,
)
dice.fit(x_train=x_train_np, y_train=y_train_np, model=model_api)
print('DiCE method fitted.')

DiCE method fitted.


In [6]:
# Pick a factual sample currently predicted as class 0 and target class 1
pred_test = model_api.predict(x_test_np)
idx = int(np.where(pred_test == 0)[0][0])
x_fact = x_test_np[idx]

example = CounterfactualExample(x=x_fact, target_class=1)
dice_result = dice.generate(example=example, model=model_api)

print(f'Factual index: {idx}')
print(f'Factual predicted class: {int(model_api.predict(x_fact)[0])}')
print(f'Counterfactual predicted class: {int(model_api.predict(dice_result.x_cf)[0])}')
print(f'Success: {dice_result.success}')
print(f'L2 distance: {dice_result.distance:.4f}')

Factual index: 0
Factual predicted class: 0
Counterfactual predicted class: 1
Success: True
L2 distance: 2.8096


## Quantitative Evaluation with Modular Metrics

In [7]:
metrics = [
    L2Proximity(),
    L1Proximity(),
    SparsityMetric(atol=1e-5),
    ValidityMetric(model=model_api),
    KNNPlausibility(x_train=x_train_np, n_neighbors=5),
]

context = {'target_class': 1}
results = {metric.name: metric.evaluate(x_fact, dice_result.x_cf, context=context) for metric in metrics}

print('DiCE metrics (modular evaluator):')
for key, value in results.items():
    print(f'  {key:15s}: {value:.6f}')

DiCE metrics (modular evaluator):
  proximity_l2   : 2.809628
  proximity_l1   : 4.233746
  sparsity       : 0.857143
  validity       : 1.000000
  plausibility   : 4.881364


In [8]:
print('CounterfactualResult fields:')
print(f'  success  : {dice_result.success}')
print(f'  distance : {dice_result.distance:.6f}')
print(f'  x_cf shape: {dice_result.x_cf.shape}')
print('  metadata :')
for key, value in dice_result.metadata.items():
    print(f'    {key}: {value}')

CounterfactualResult fields:
  success  : True
  distance : 2.809628
  x_cf shape: (14,)
  metadata :
    target_class: 1
    n_valid: 1
    n_candidates: 4
    optimization_loss: -1.6744389235973358
    iterations: 300
    mean_target_proba: 0.2336232215166092
